In [ ]:
# Example 1: Verify the primary public API surface exists.
from opentelemetry.instrumentation.requests import RequestsInstrumentor

instrumentor = RequestsInstrumentor()
print(f"instrumentor type: {type(instrumentor).__name__}")
print(f"instrumentation_dependencies: {instrumentor.instrumentation_dependencies()}")

# Class must expose the standard BaseInstrumentor lifecycle methods.
for attr in ('instrument', 'uninstrument', 'instrumentation_dependencies'):
    assert hasattr(instrumentor, attr), f"missing attr: {attr}"
print("RequestsInstrumentor exposes expected API")


In [ ]:
# Example 2: instrument() + uninstrument() lifecycle is idempotent.
from opentelemetry.instrumentation.requests import RequestsInstrumentor

instrumentor = RequestsInstrumentor()

# Make sure we start from a clean state.
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()

instrumentor.instrument()
print(f"after instrument: is_instrumented_by_opentelemetry = {instrumentor.is_instrumented_by_opentelemetry}")
assert instrumentor.is_instrumented_by_opentelemetry is True

instrumentor.uninstrument()
print(f"after uninstrument: is_instrumented_by_opentelemetry = {instrumentor.is_instrumented_by_opentelemetry}")
assert instrumentor.is_instrumented_by_opentelemetry is False

In [ ]:
# Example 3: instrumented requests.Session.send is patched + restored.
import requests
from opentelemetry.instrumentation.requests import RequestsInstrumentor

original_send = requests.Session.send

instrumentor = RequestsInstrumentor()
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()

instrumentor.instrument()
patched_send = requests.Session.send
assert patched_send is not original_send, "Session.send should be patched after instrument()"
print("Session.send patched after instrument()")

instrumentor.uninstrument()
restored_send = requests.Session.send
assert restored_send is original_send, "Session.send should be restored after uninstrument()"
print("Session.send restored after uninstrument()")

In [ ]:
# Example 4: TracerProvider + span capture for an in-process requests call.
import socket
import requests
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.requests import RequestsInstrumentor

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

instrumentor = RequestsInstrumentor()
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()
instrumentor.instrument(tracer_provider=provider)

# Hit a loopback port nothing listens on (bound, read, released); the span is still emitted on connection error.
_probe = socket.socket()
_probe.bind(("127.0.0.1", 0))
closed_port = _probe.getsockname()[1]
_probe.close()
try:
    requests.get(f"http://127.0.0.1:{closed_port}/", timeout=0.1)
except requests.exceptions.ConnectionError as e:
    print(f"connection refused as expected -> {type(e).__name__}")
else:
    raise AssertionError(f"GET to closed port {closed_port} unexpectedly succeeded")

spans = exporter.get_finished_spans()
print(f"captured {len(spans)} span(s); names: {[s.name for s in spans]}")
instrumentor.uninstrument()
assert len(spans) >= 1, "expected at least one span from the instrumented requests call"
assert any(s.status.status_code == trace.StatusCode.ERROR for s in spans), [s.status for s in spans]

In [ ]:
# Example 5: request/response hooks fire with expected arg shape.
import socket
import requests
from opentelemetry.instrumentation.requests import RequestsInstrumentor

calls = {"request": 0, "response": 0}

def request_hook(span, request_obj):
    calls["request"] += 1

def response_hook(span, request_obj, response):
    calls["response"] += 1

instrumentor = RequestsInstrumentor()
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()
instrumentor.instrument(request_hook=request_hook, response_hook=response_hook)

# A loopback port nothing listens on (bound, read, released): the request hook runs, then the connect is refused.
_probe = socket.socket()
_probe.bind(("127.0.0.1", 0))
closed_port = _probe.getsockname()[1]
_probe.close()
try:
    requests.get(f"http://127.0.0.1:{closed_port}/", timeout=0.1)
except requests.exceptions.ConnectionError as e:
    print(f"connection refused as expected -> {type(e).__name__}")
else:
    raise AssertionError(f"GET to closed port {closed_port} unexpectedly succeeded")

instrumentor.uninstrument()
print(f"hook call counts: {calls}")
assert calls["request"] >= 1, "request_hook should have fired at least once"